# Transfer-Learning Intervals: Global vs Grouped Pooling

Temporary notebook to compare pooled conformal intervals for new transfer-learning items:
- global pooling over all calibrated ids
- grouped pooling with categorical static features


In [1]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals, generate_daily_series

try:
    from datasetsforecast.m5 import M5
    HAS_M5 = True
except Exception:
    HAS_M5 = False

HAS_M5


True

In [2]:
h = 14

if HAS_M5:
    y_df, _, s_df = M5.load(directory="data")
    y_df["ds"] = pd.to_datetime(y_df["ds"])
    df = y_df.merge(s_df, on="unique_id", how="left")
    all_ids = df["unique_id"].drop_duplicates().tolist()
    base_ids = set(all_ids[:80])
    transfer_ids = set(all_ids[60:120])
    static_features = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
    interval_groupby = ["state_id", "cat_id"]
    base_df = df[df["unique_id"].isin(base_ids)].copy()
    transfer_all = df[df["unique_id"].isin(transfer_ids)].copy()
else:
    # Synthetic fallback with two categorical static features and group-dependent scale
    def add_groups(tmp):
        out = tmp.copy()
        idx = out["unique_id"].astype(str).str.extract(r"(\d+)").astype(int)[0]
        out["cat0"] = np.where(idx % 2 == 0, "A", "B")
        out["cat1"] = np.where((idx // 2) % 2 == 0, "X", "Y")
        out["y"] = out["y"] * np.where(out["cat0"] == "B", 8.0, 1.0)
        out["cat0"] = out["cat0"].astype("category")
        out["cat1"] = out["cat1"].astype("category")
        return out

    base_df = add_groups(generate_daily_series(4, 120, 120, equal_ends=True, seed=0))
    transfer_all = add_groups(generate_daily_series(6, 120, 120, equal_ends=True, seed=1))
    base_ids = set(base_df["unique_id"].drop_duplicates())
    transfer_ids = set(transfer_all["unique_id"].drop_duplicates())
    static_features = ["cat0", "cat1"]
    interval_groupby = ["cat0", "cat1"]

transfer_valid = transfer_all.groupby("unique_id", observed=True).tail(h)
transfer_train = transfer_all.drop(transfer_valid.index)

# Keep native id dtype for predict(ids=...)
base_ids_set = set(base_df["unique_id"])
transfer_ids_set = set(transfer_train["unique_id"])
existing_ids = sorted(transfer_ids_set & base_ids_set)
new_ids = sorted(transfer_ids_set - base_ids_set)

# Holdout on base data for normal (non-transfer) conformal evaluation on old ids only.
base_old_valid = base_df[base_df["unique_id"].isin(existing_ids)].groupby("unique_id", observed=True).tail(h)
base_old_train = base_df.drop(base_old_valid.index)

len(base_df["unique_id"].unique()), len(transfer_train["unique_id"].unique()), len(existing_ids), len(new_ids)


(80, 60, 20, 40)

In [3]:
fit_kwargs = dict(
    static_features=static_features,
    prediction_intervals=PredictionIntervals(n_windows=12, h=h, 
                                             method="conformal_error",)
    
)

# 1) Transfer-learning + global pooled fallback
fcst_global = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq="D",
    lags=[1, 7, 28],
    num_threads=1,
).fit(base_df, **fit_kwargs)

with warnings.catch_warnings(record=True) as caught_global:
    warnings.simplefilter("always")
    preds_global = fcst_global.predict(h=h, new_df=transfer_train, level=[80])

# 2) Transfer-learning + grouped pooled fallback
fcst_grouped = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq="D",
    lags=[1, 7, 28],
    num_threads=1,
).fit(base_df, **fit_kwargs)

with warnings.catch_warnings(record=True) as caught_grouped:
    warnings.simplefilter("always")
    preds_grouped = fcst_grouped.predict(
        h=h,
        new_df=transfer_train,
        level=[80],
        interval_groupby=interval_groupby,
    )

# 3) Normal conformal prediction on old ids only (no transfer learning)
fcst_normal_old = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq="D",
    lags=[1, 7, 28],
    num_threads=1,
).fit(base_old_train, **fit_kwargs)

with warnings.catch_warnings(record=True) as caught_normal_old:
    warnings.simplefilter("always")
    preds_normal_old = fcst_normal_old.predict(
        h=h,
        level=[80],
    )

# Guard: fail early with a clear error if intervals are missing.
if not any(str(c).endswith("-lo-80") for c in preds_normal_old.columns):
    raise ValueError(
        "preds_normal_old has no interval columns. "
        f"Columns: {list(preds_normal_old.columns)}. "
        "Re-run from cell 2/3 and make sure PredictionIntervals was used in fit."
    )

print("global warnings:", [str(w.message) for w in caught_global])
print("grouped warnings:", [str(w.message) for w in caught_grouped])
print("normal-old warnings:", [str(w.message) for w in caught_normal_old])


[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero
global warnings: ['Prediction intervals were calibrated on a different set of series than the current transfer-learning state. Falling back to pooled conformity scores.']
grouped warnings: ['Prediction intervals were calibrated on a different set of series than the current transfer-learning state. Falling back to pooled conformity scores.']
normal-old warnings: []


In [4]:
def get_interval_cols(df, level=80):
    lo_suffix = f"-lo-{level}"
    hi_suffix = f"-hi-{level}"
    lo_candidates = [c for c in df.columns if str(c).endswith(lo_suffix)]
    hi_candidates = [c for c in df.columns if str(c).endswith(hi_suffix)]
    if not lo_candidates or not hi_candidates:
        raise ValueError(
            "Could not find interval columns for level 80. "
            f"Available columns: {list(df.columns)}"
        )
    lo_col = lo_candidates[0]
    model_name = lo_col[: -len(lo_suffix)]
    hi_col = f"{model_name}{hi_suffix}"
    if hi_col not in df.columns:
        raise ValueError(
            f"Found lo column {lo_col} but missing matching hi column {hi_col}. "
            f"Available columns: {list(df.columns)}"
        )
    return lo_col, hi_col


eval_global = preds_global.merge(
    transfer_valid[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left",
)
eval_grouped = preds_grouped.merge(
    transfer_valid[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left",
)
eval_normal_old = preds_normal_old.merge(
    base_old_valid[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="left",
)


def summarize(eval_df, setting, cohorts):
    lo_col, hi_col = get_interval_cols(eval_df, level=80)
    rows = []
    for cohort, ids in cohorts:
        subset = eval_df[eval_df["unique_id"].isin(ids)].copy()
        coverage = ((subset["y"] >= subset[lo_col]) & (subset["y"] <= subset[hi_col])).mean()
        avg_width = (subset[hi_col] - subset[lo_col]).mean()
        rows.append(
            {
                "setting": setting,
                "cohort": cohort,
                "n_ids": len(ids),
                "n_rows": len(subset),
                "coverage_80": coverage,
                "avg_interval_width_80": avg_width,
                "interval_cols": f"{lo_col} | {hi_col}",
            }
        )
    return pd.DataFrame(rows)


results = pd.concat(
    [
        summarize(
            eval_global,
            "transfer_global_pooling",
            [("existing_items", existing_ids), ("new_items", new_ids)],
        ),
        summarize(
            eval_grouped,
            "transfer_grouped_pooling",
            [("existing_items", existing_ids), ("new_items", new_ids)],
        ),
        summarize(
            eval_normal_old,
            "normal_conformal_old_ids",
            [("existing_items", existing_ids)],
        ),
    ],
    ignore_index=True,
)

results.sort_values(["cohort", "setting"]).reset_index(drop=True)


,setting,cohort,n_ids,n_rows,coverage_80,avg_interval_width_80,interval_cols
0,normal_conformal_old_ids,existing_items,20,280,0.667857,1.404087,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
1,transfer_global_pooling,existing_items,20,280,0.903571,3.307630,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
2,transfer_grouped_pooling,existing_items,20,280,0.896429,3.151570,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
3,transfer_global_pooling,new_items,40,560,0.739286,3.307630,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
4,transfer_grouped_pooling,new_items,40,560,0.732143,3.151570,LGBMRegressor-lo-80 | LGBMRegressor-hi-80
